# EEEM068: Industrial Waste Classification
## Notebook 03 — ResNet50 Training

**Author:** Scott Lewis  
**Date:** March 2026  

---

### Objectives

This notebook trains and evaluates a ResNet50 backbone on WaRP-C:

1. Load pretrained ResNet50 and replace classification head
2. Train using Phase 1 reference settings (lr=1e-4, bs=64, 30 epochs)
3. Monitor training with loss and accuracy curves
4. Evaluate on test set with full metrics
5. Analyse per-class performance

---

## 0. Setup & Imports

In [ ]:
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score
)
from PIL import Image

# import pipeline from notebook 02
import sys
sys.path.insert(0, str(Path('.')))

# seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

plt.rcParams['figure.dpi'] = 120

# paths
PROJECT_ROOT = Path('../../')
DATASET_DIR  = Path('dataset')
OUTPUT_DIR   = Path('outputs/resnet50')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# class mapping
with open(DATASET_DIR / 'class_mapping.json') as f:
    mapping = json.load(f)
label_to_class = {int(k): v for k, v in mapping['label_to_class'].items()}
n_classes      = mapping['n_classes']
class_names    = [label_to_class[i] for i in range(n_classes)]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Hyperparameters

Phase 1 reference run settings (R8 in the grid).

In [ ]:
# =============================================================================
# HYPERPARAMETERS — Phase 1 reference run (R8)
# Change these to run different grid configurations
# =============================================================================

LEARNING_RATE = 1e-4
BATCH_SIZE    = 64
EPOCHS        = 30
PRETRAINED    = True
STAGED_LR     = False   # set True for Phase 2
BASE_LR_MULT  = 0.1     # backbone lr multiplier when staged_lr=True
T_MAX         = 20      # cosine annealing cycle
WEIGHT_DECAY  = 5e-4
IMG_SIZE      = (224, 224)

# augmentation
BRIGHTNESS    = 0.2
CONTRAST      = 0.2
SATURATION    = 0.2
HUE           = 0.1

RUN_NAME = f'resnet50_phase1_lr{LEARNING_RATE}_bs{BATCH_SIZE}'
print(f'Run: {RUN_NAME}')
print(f'  LR: {LEARNING_RATE} | BS: {BATCH_SIZE} | Epochs: {EPOCHS} | Pretrained: {PRETRAINED}')

## 2. Data Pipeline

In [ ]:
class WaRPDataset(Dataset):
    def __init__(self, csv_path, project_root, transform=None):
        self.df           = pd.read_csv(csv_path)
        self.project_root = Path(project_root)
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img      = Image.open(self.project_root / row['filepath']).convert('RGB')
        label    = int(row['label'])
        if self.transform:
            img  = self.transform(img)
        return img, label

    def get_class_weights(self):
        counts  = self.df['label'].value_counts().sort_index()
        weights = 1.0 / (counts.values + 1e-6)
        return torch.FloatTensor(weights)


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(BRIGHTNESS, CONTRAST, SATURATION, HUE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

pin     = torch.cuda.is_available()
workers = 0 if os.name == 'nt' else 4

train_ds = WaRPDataset(DATASET_DIR / 'train.csv', PROJECT_ROOT, train_tf)
val_ds   = WaRPDataset(DATASET_DIR / 'val.csv',   PROJECT_ROOT, val_tf)
test_ds  = WaRPDataset(DATASET_DIR / 'test.csv',  PROJECT_ROOT, val_tf)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=workers, pin_memory=pin)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=workers, pin_memory=pin)
test_loader  = DataLoader(test_ds,  32,         shuffle=False, num_workers=workers, pin_memory=pin)

class_weights = train_ds.get_class_weights()

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Train batches: {len(train_loader)}')

## 3. Model

In [ ]:
weights = 'IMAGENET1K_V1' if PRETRAINED else None
model   = models.resnet50(weights=weights)

# replace classification head for 28 classes
model.fc = nn.Linear(model.fc.in_features, n_classes)
model    = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ResNet50 loaded (pretrained={PRETRAINED})')
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

In [ ]:
# loss, optimizer, scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

if STAGED_LR:
    # backbone at lower lr, head at full lr
    backbone_params = [p for n, p in model.named_parameters() if 'fc' not in n]
    head_params     = [p for n, p in model.named_parameters() if 'fc' in n]
    param_groups    = [
        {'params': backbone_params, 'lr': LEARNING_RATE * BASE_LR_MULT},
        {'params': head_params,     'lr': LEARNING_RATE}
    ]
    print(f'Staged LR — backbone: {LEARNING_RATE * BASE_LR_MULT:.2e} | head: {LEARNING_RATE:.2e}')
else:
    param_groups = model.parameters()
    print(f'Uniform LR — {LEARNING_RATE:.2e}')

optimizer = torch.optim.Adam(param_groups, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

## 4. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        preds       = outputs.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / total, correct / total, f1, all_preds, all_labels


print('Training functions defined')

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_f1    = 0.0
best_epoch = 0
ckpt_path  = OUTPUT_DIR / 'best_model.pth'

print(f'Starting training — {EPOCHS} epochs')
print(f'{"-"*60}')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, val_preds, val_labels = val_epoch(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    elapsed = time.time() - t0
    marker  = ' *' if val_f1 > best_f1 else ''

    print(f'Epoch {epoch:>3}/{EPOCHS} | '
          f'train loss: {train_loss:.4f} acc: {train_acc:.4f} | '
          f'val loss: {val_loss:.4f} acc: {val_acc:.4f} f1: {val_f1:.4f} | '
          f'{elapsed:.0f}s{marker}')

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_epoch = epoch
        torch.save(model.state_dict(), ckpt_path)

print(f'{"-"*60}')
print(f'Training complete. Best val F1: {best_f1:.4f} at epoch {best_epoch}')

## 5. Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'],   label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], label='Train')
axes[1].plot(epochs_range, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(epochs_range, history['val_f1'], color='green', label='Val F1')
axes[2].axvline(best_epoch, color='red', linestyle='--', label=f'Best epoch ({best_epoch})')
axes[2].set_title('Validation F1 (macro)')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.suptitle(f'ResNet50 Training Curves — {RUN_NAME}', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## 6. Test Evaluation

In [ ]:
# load best checkpoint
model.load_state_dict(torch.load(ckpt_path, map_location=device))
print(f'Loaded best checkpoint from epoch {best_epoch}')

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images  = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


preds, labels, probs = evaluate(model, test_loader, device)

# compute metrics
accuracy  = float(np.mean(labels == preds))
f1_macro  = float(f1_score(labels, preds, average='macro',    zero_division=0))
f1_w      = float(f1_score(labels, preds, average='weighted', zero_division=0))
prec      = float(precision_score(labels, preds, average='macro', zero_division=0))
rec       = float(recall_score(labels, preds, average='macro',    zero_division=0))

one_hot = np.eye(n_classes)[labels]
try:
    auc = float(roc_auc_score(one_hot, probs, average='macro', multi_class='ovr'))
except:
    auc = None
try:
    mAP = float(average_precision_score(one_hot, probs, average='macro'))
except:
    mAP = None

print(f'\n{"="*45}')
print(f'  TEST RESULTS — {RUN_NAME}')
print(f'{"="*45}')
print(f'  Accuracy:          {accuracy:.4f}')
print(f'  F1 (macro):        {f1_macro:.4f}')
print(f'  F1 (weighted):     {f1_w:.4f}')
print(f'  Precision (macro): {prec:.4f}')
print(f'  Recall (macro):    {rec:.4f}')
print(f'  AUC:               {auc:.4f}' if auc else '  AUC:               N/A')
print(f'  mAP:               {mAP:.4f}' if mAP else '  mAP:               N/A')
print(f'{"="*45}')

# save metrics
results = {
    'run_name': RUN_NAME, 'accuracy': accuracy,
    'f1_macro': f1_macro, 'f1_weighted': f1_w,
    'precision_macro': prec, 'recall_macro': rec,
    'auc': auc, 'mAP': mAP
}
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved: test_metrics.json')

## 7. Confusion Matrix

In [ ]:
cm  = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names)
ax.set_title(f'Confusion Matrix — ResNet50 ({RUN_NAME})', fontsize=13)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

## 8. Per-Class Performance

In [ ]:
report = classification_report(labels, preds, target_names=class_names,
                                output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T.iloc[:-3]  # drop avg rows
report_df  = report_df.sort_values('f1-score', ascending=False)

print('Per-class F1 scores (sorted):')
print(report_df[['precision','recall','f1-score','support']].round(3).to_string())

# plot per-class F1
fig, ax = plt.subplots(figsize=(10, 8))
colours = ['#d95f02' if f < 0.3 else '#1b9e77' if f > 0.7 else '#7570b3'
           for f in report_df['f1-score']]
ax.barh(report_df.index, report_df['f1-score'], color=colours, edgecolor='white')
ax.axvline(f1_macro, color='red', linestyle='--', label=f'Macro F1 ({f1_macro:.3f})')
ax.set_xlabel('F1 Score')
ax.set_title(f'Per-class F1 — ResNet50 ({RUN_NAME})')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: per_class_f1.png')

## 9. Summary

Observations from this run should be recorded in your weekly log and compared against other model results in `04_results_analysis.ipynb`.

In [ ]:
# hardest classes (lowest F1)
hardest = report_df.tail(5)
print('5 hardest classes (lowest F1):')
for cls, row in hardest.iterrows():
    print(f'  {cls:<35} F1: {row["f1-score"]:.3f}')

# easiest classes (highest F1)
easiest = report_df.head(5)
print('\n5 easiest classes (highest F1):')
for cls, row in easiest.iterrows():
    print(f'  {cls:<35} F1: {row["f1-score"]:.3f}')